In [ ]:
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
import numpy as np
import pickle
import json
from pathlib import Path

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

## Define Calculation Function

In [ ]:
def calculate_csd(sim_path):
    """Calculates cloud statistics for a given simulation."""

    # model grid parameters
    dx, dy, dz = 25, 25, 25 # m
    grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3

    # tunable parameters
    # the highest cloud base level (above domain mean cloud base level)
    # allowed to be considered an attached cloud
    cbl_gap = 5
    # cloud liquid water mixing ratio criterion
    # for determining domain mean cloud base
    qclm_crit = 1.0e-6 # g/kg

    sim_path = Path(sim_path)

    with open(sim_path / 'uninterrupted_large_clouds.json', 'r') as f:
        ul_clouds = json.load(f)
    ul_clouds_list = list(ul_clouds.keys())
    nc = len(ul_clouds_list)
    id_list = np.asarray(ul_clouds_list, dtype=np.int32)

    with open(sim_path / 'pkl/cloud_all_af.pkl', 'rb') as f:
        cloud_areas = pickle.load(f)

    cloud_volumes = cloud_areas.sum(axis=2)
    cloud_times = (cloud_volumes > 0).sum(axis=1)
    total_cloud_volumes = cloud_volumes.sum(axis=1) * grid_vol # in km**3   

    cbl_c = np.argmax(cloud_areas > 0, axis=2)
    # min cloud base and max cloud top for each cloud
    mcbl_c = np.zeros(nc, dtype=np.int32)
    mctl_c = np.zeros(nc, dtype=np.int32)
    for i in range(nc):
        a = cloud_areas[i,:,:].sum(axis=0)
        mcbl_c[i] = ((np.where(a != 0))[0]).min()
        mctl_c[i] = ((np.where(a != 0))[0]).max()

    with open(sim_path / 'pkl/qclm.pkl', 'rb') as f:
        qclm = np.asarray(pickle.load(f))
    mcbl = np.argmax(qclm > qclm_crit)

    attached_c = (cloud_volumes > 0) & (cbl_c < (mcbl + cbl_gap) )
    attached = attached_c.sum(axis=1) > 0
    attached_clouds = id_list[attached]
    attached_ind = np.asarray(np.nonzero(np.isin(id_list, attached_clouds, assume_unique=True))[0])

    with open(sim_path / 'pkl/plume_all_mf.pkl', 'rb') as f:
        plume_mfs = pickle.load(f)
    plume_mfs = plume_mfs*dx*dy # now in kg/s unit

    def calc_mean_mf(plume_mfs, mcbl, attached_c):
        nc, nt, nz = plume_mfs.shape
        mf = np.zeros(nc, dtype=float)
        mf_t = np.zeros((nc, nt), dtype=float)
        for i in range(nc):
            a = attached_c[i,:]
            for t in range(nt):
                # average over 3 levels around cloud base
                mf_t[i,t] = plume_mfs[i, t, mcbl-1:mcbl+2].mean()
            if a.sum() != 0:
                mf[i] = (a * mf_t[i,:]).sum()/a.sum()
        return mf

    mean_cb_mf = calc_mean_mf(plume_mfs, mcbl, attached_c)
    mean_cb_mf_c = np.where(mean_cb_mf > 1.0, mean_cb_mf, 1.1)
    clipped_mf = mean_cb_mf_c[attached_ind]

    return clipped_mf, mean_cb_mf[attached_ind], total_cloud_volumes[attached_ind], cloud_times[attached_ind], mcbl_c[attached_ind], mctl_c[attached_ind], attached_ind

## Perform Calculations

In [ ]:
stat_ehe18 = calculate_csd('bomex_25m_ehe18_r20251009/')
stat_ctl = calculate_csd('bomex_25m_r20251009/')
with open('bomex_25m_ehe18_r20251009/pkl/csd_stats.pkl', 'wb') as f:
    pickle.dump(stat_ehe18, f)
with open('bomex_25m_r20251009/pkl/csd_stats.pkl', 'wb') as f:
    pickle.dump(stat_ctl, f)

In [ ]:
stat_ehe21 = calculate_csd('bomex_25m_ehe21_r20251107/')
stat_ehe22 = calculate_csd('bomex_25m_ehe22_r20251107/')
with open('bomex_25m_ehe21_r20251107/pkl/csd_stats.pkl', 'wb') as f:
    pickle.dump(stat_ehe21, f)
with open('bomex_25m_ehe22_r20251107/pkl/csd_stats.pkl', 'wb') as f:
    pickle.dump(stat_ehe22, f)